# Домашнее задание 1 — Python
Запустить ячейки сверху вниз. `transactions_dirty.csv` и
`transactions_dirty.json` должны лежать рядом с ноутбуком.
В Colab нужно загрузить оба файла в раздел «Файлы».

In [1]:
import csv
import json
import math
import re
from datetime import datetime
from pathlib import Path

## 1. Класс Account

In [2]:
class Account:
    _account_counter = 1000
    account_type = "account"
    allowed_operations = {"deposit", "withdraw"}
    withdrawal_fraction = 1

    def __init__(self, account_holder, balance=0):
        pattern = r"(?:[А-ЯЁ][а-яё]+ [А-ЯЁ][а-яё]+|[A-Z][a-z]+ [A-Z][a-z]+)"
        if not isinstance(account_holder, str) or not re.fullmatch(
            pattern, account_holder
        ):
            raise ValueError("Имя должно быть в формате: Имя Фамилия.")
        self._balance = self._check_amount(balance, allow_zero=True)
        self._initial_balance = self._balance
        self.holder = account_holder
        self.account_number = f"ACC-{Account._account_counter:04d}"
        Account._account_counter += 1
        self.operations_history = []

    @staticmethod
    def _check_amount(amount, allow_zero=False):
        if isinstance(amount, bool) or not isinstance(amount, (int, float)):
            raise ValueError("Сумма должна быть числом.")
        if not math.isfinite(amount) or amount < 0:
            raise ValueError("Сумма должна быть конечной и неотрицательной.")
        amount = round(amount, 2)
        if amount == 0 and not allow_zero:
            raise ValueError("Сумма должна быть положительной.")
        return amount

    def _record(self, operation, amount, status):
        self.operations_history.append({
            "account_number": self.account_number,
            "account_type": self.account_type,
            "date": datetime.now(),
            "operation": operation,
            "amount": amount,
            "balance_after": self._balance,
            "status": status,
        })

    def deposit(self, amount):
        amount = self._check_amount(amount)
        self._balance = self._check_amount(self._balance + amount)
        self._record("deposit", amount, "success")

    def withdraw(self, amount):
        amount = self._check_amount(amount)
        if amount > self._balance * self.withdrawal_fraction:
            self._record("withdraw", amount, "fail")
            return False
        self._balance = round(self._balance - amount, 2)
        self._record("withdraw", amount, "success")
        return True

    def get_balance(self):
        return self._balance

    def get_history(self):
        return [
            dict(row, date=row["date"].isoformat(sep=" "))
            for row in self.operations_history
        ]

    def get_large_operations(self, n=5, min_amount=500):
        # последние n успешных операций на сумму от min_amount
        if type(n) is not int or n < 0:
            raise ValueError("n должно быть целым неотрицательным числом.")
        min_amount = self._check_amount(min_amount, allow_zero=True)
        rows = [row for row in self.get_history()
                if row["amount"] >= min_amount and row["status"] == "success"]
        return sorted(rows, key=lambda row: row["date"], reverse=True)[:n]

    @staticmethod
    def _parse_date(value):
        if isinstance(value, datetime):
            return value
        for pattern in ("%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M"):
            try:
                return datetime.strptime(value, pattern)
            except ValueError:
                pass
        raise ValueError("Неверная дата.")

    def clean_history(self, records=None, source_account_number=None):
        # отбрасывает ошибки и дубли, пересчитывает остаток
        if records is None:
            records = self.operations_history
        if not isinstance(records, list):
            raise ValueError("История должна быть списком записей.")
        source = source_account_number or self.account_number
        valid = []
        seen = set()
        for row in records:
            try:
                if not isinstance(row, dict):
                    continue
                if row["account_number"] != source:
                    continue
                if row["account_type"] != self.account_type:
                    continue
                if row["operation"] not in self.allowed_operations:
                    continue
                if row["status"] not in ("success", "fail"):
                    continue
                row = row.copy()
                row["date"] = self._parse_date(row["date"])
                for field in ("amount", "balance_after"):
                    value = row[field]
                    if isinstance(value, str):
                        value = float(value)
                    allow_zero = (
                        field == "balance_after"
                        or row["operation"] == "interest"
                    )
                    row[field] = self._check_amount(value, allow_zero)
                key = tuple(row[field] for field in (
                    "date", "operation", "amount", "balance_after", "status"
                ))
                if key not in seen:
                    seen.add(key)
                    row["account_number"] = self.account_number
                    valid.append(row)
            except (KeyError, TypeError, ValueError):
                continue

        # старые остатки учитывают удалённые операции, поэтому считаем заново
        balance = self._initial_balance
        cleaned = []
        for row in sorted(valid, key=lambda row: row["date"]):
            new_balance = balance
            if row["status"] == "success":
                if row["operation"] == "withdraw":
                    if row["amount"] > balance * self.withdrawal_fraction:
                        continue
                    new_balance -= row["amount"]
                else:
                    new_balance += row["amount"]
            try:
                balance = self._check_amount(new_balance, allow_zero=True)
            except ValueError:
                continue
            row["balance_after"] = balance
            cleaned.append(row)
        return cleaned

    def load_history(self, file_path, source_account_number=None):
        # заменяет историю операциями выбранного счёта из файла
        path = Path(file_path)
        with path.open(encoding="utf-8-sig", newline="") as file:
            if path.suffix.lower() == ".csv":
                rows = list(csv.DictReader(file))
            elif path.suffix.lower() == ".json":
                rows = json.load(file)
            else:
                raise ValueError("Нужен файл CSV или JSON.")
        source = source_account_number or self.account_number
        if not isinstance(rows, list) or not any(
            isinstance(row, dict) and row.get("account_number") == source
            for row in rows
        ):
            raise ValueError("Не найден список операций выбранного счёта.")
        self.operations_history = self.clean_history(rows, source)
        self._balance = (
            self.operations_history[-1]["balance_after"]
            if self.operations_history else self._initial_balance
        )
        return len(self.operations_history)

## 2. Наследование и примеры операций

In [3]:
class CheckingAccount(Account):
    account_type = "checking"


class SavingsAccount(Account):
    account_type = "savings"
    allowed_operations = {"deposit", "withdraw", "interest"}
    withdrawal_fraction = 0.5

    def withdraw(self, amount):
        # общая проверка использует лимит этого класса — 50%
        return super().withdraw(amount)

    def apply_interest(self, rate):
        rate = self._check_amount(rate, allow_zero=True)
        amount = self._check_amount(self._balance * rate / 100, True)
        self._balance = self._check_amount(self._balance + amount, True)
        self._record("interest", amount, "success")
        return amount

In [4]:
def show_history(history):
    for row in history:
        print(f"{row['date'][:19]} | {row['operation']:8} | "
              f"сумма {row['amount']:.2f} | "
              f"баланс {row['balance_after']:.2f} | {row['status']}")


checking = CheckingAccount("Иван Иванов", 1000)
savings = SavingsAccount("Anna Smith", 2000)
checking.deposit(500)
checking.withdraw(300)
checking.withdraw(2000)
savings.apply_interest(7)
savings.withdraw(1200)
savings.withdraw(1070)

for account in (checking, savings):
    print(account.account_number, account.holder, account.account_type)
    show_history(account.get_history())

print("Последние крупные операции:")
show_history(savings.get_large_operations(n=2, min_amount=500))

ACC-1000 Иван Иванов checking
2026-10-04 17:00:16 | deposit  | сумма 500.00 | баланс 1500.00 | success
2026-10-04 17:00:16 | withdraw | сумма 300.00 | баланс 1200.00 | success
2026-10-04 17:00:16 | withdraw | сумма 2000.00 | баланс 1200.00 | fail
ACC-1001 Anna Smith savings
2026-10-04 17:00:16 | interest | сумма 140.00 | баланс 2140.00 | success
2026-10-04 17:00:16 | withdraw | сумма 1200.00 | баланс 2140.00 | fail
2026-10-04 17:00:16 | withdraw | сумма 1070.00 | баланс 1070.00 | success
Последние крупные операции:
2026-10-04 17:00:16 | withdraw | сумма 1070.00 | баланс 1070.00 | success


In [5]:
try:
    checking.deposit(-100)
except ValueError as error:
    print(error)

try:
    Account("иван Иванов")
except ValueError as error:
    print(error)

Сумма должна быть конечной и неотрицательной.
Имя должно быть в формате: Имя Фамилия.


## 3. Загрузка и очистка истории
Методы загрузки и очистки наследуются от Account, начальный баланс равен 0 — значению по умолчанию. Ошибки, дубли и недопустимые снятия исключаются, остатки пересчитываются

In [6]:
account_specs = [
    ("ACC-100001", CheckingAccount, "Иван Иванов"),
    ("ACC-100002", SavingsAccount, "Анна Смирнова"),
    ("ACC-100003", CheckingAccount, "John Smith"),
    ("ACC-100004", SavingsAccount, "Мария Петрова"),
]
imported = {}
for extension in ("csv", "json"):
    for number, account_class, holder in account_specs:
        account = account_class(holder)
        count = account.load_history(
            f"transactions_dirty.{extension}", source_account_number=number
        )
        imported[extension, number] = account
        print(extension, number, "операций:", count,
              "баланс:", account.get_balance())

csv ACC-100001 операций: 17 баланс: 3664.0
csv ACC-100002 операций: 17 баланс: 2883.24
csv ACC-100003 операций: 13 баланс: 4646.0
csv ACC-100004 операций: 19 баланс: 8057.25
json ACC-100001 операций: 17 баланс: 3664.0
json ACC-100002 операций: 17 баланс: 2883.24
json ACC-100003 операций: 13 баланс: 4646.0
json ACC-100004 операций: 19 баланс: 8057.25
